# 05 — Association Rules Mining (ARM) — CAMS Air Quality × Health

Discovers non-linear combinations of **air quality conditions** that co-occur with elevated health emergency demand.

**This run uses CAMS data only** (PM2.5, PM10, NO₂, O₃, SO₂) — no ERA5 required.  
COVID period (2020-03-15 → 2021-06-30) is excluded from all analyses.

**Approach**:
1. Load CAMS netCDF files, resample to daily means, merge with health data
2. Discretise AQ variables (WHO-threshold bins) and health outcomes (quantile bins)
3. Apply FP-Growth to mine frequent itemsets
4. Filter rules where the consequent is an elevated/extreme health outcome
5. Stratify by season; stability filter (≥ 2 strata)
6. Negative control validation (stabbing/traffic lift should be ≈ 1)

**Input**: `data/raw/cams/cams_berlin_YYYY_MM.nc`, `data/processed/daily_city_health.csv`  
**Output**: `data/processed/arm_rules_cams.csv`, `arm_rules_stable_cams.csv`, plots

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from mlxtend.frequent_patterns import fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

PROC_DIR = Path('../data/processed')

print('Libraries loaded')

In [ ]:
# ── Load preprocessed CAMS daily means (2025) ────────────────────────────────
cams_df = pd.read_csv(PROC_DIR / 'cams_daily_2025.csv', parse_dates=['date'])
print(f'CAMS city-wide: {cams_df.shape}  |  {cams_df.date.min().date()} → {cams_df.date.max().date()}')
print(cams_df[['pm25','pm10','no2','o3','so2']].describe().round(2))

In [ ]:
# ── Load health data, filter to 2025, merge ───────────────────────────────────
health_df = pd.read_csv(PROC_DIR / 'daily_city_health.csv', parse_dates=['date'])
health_df = health_df[health_df['date'].dt.year == 2025].reset_index(drop=True)

df = health_df.merge(cams_df, on='date', how='inner')
df = df.sort_values('date').reset_index(drop=True)

print(f'Health 2025:  {health_df.shape[0]} days')
print(f'After merge:  {df.shape}  |  {df.date.min().date()} → {df.date.max().date()}')
print(f'AQ columns:   {[c for c in ["pm25","pm10","no2","o3","so2"] if c in df.columns]}')

# ── Air quality bins (WHO + EEA threshold-based) ────────────────────────────
# All concentrations in µg/m³
CLIMATE_BINS = {
    'pm25': {
        'bins':   [-np.inf,  5,  10,  25,  50, np.inf],
        'labels': ['pm25_very_good', 'pm25_good', 'pm25_moderate',
                   'pm25_unhealthy', 'pm25_hazardous'],
    },
    'pm10': {
        'bins':   [-np.inf, 10,  20,  40,  80, np.inf],
        'labels': ['pm10_very_good', 'pm10_good', 'pm10_moderate',
                   'pm10_unhealthy', 'pm10_hazardous'],
    },
    'no2': {
        'bins':   [-np.inf, 10,  20,  40,  80, np.inf],
        'labels': ['no2_low', 'no2_moderate', 'no2_elevated', 'no2_high', 'no2_very_high'],
    },
    'o3': {
        'bins':   [-np.inf, 60,  100, 140, 180, np.inf],
        'labels': ['o3_low', 'o3_moderate', 'o3_elevated', 'o3_high', 'o3_very_high'],
    },
    'so2': {
        'bins':   [-np.inf, 20,  50, 100, 200, np.inf],
        'labels': ['so2_low', 'so2_moderate', 'so2_elevated', 'so2_high', 'so2_very_high'],
    },
}

discr = pd.DataFrame({'date': df['date']})
for col, spec in CLIMATE_BINS.items():
    if col in df.columns:
        discr[col] = pd.cut(df[col], bins=spec['bins'], labels=spec['labels'])

available_climate = [c for c in CLIMATE_BINS if c in df.columns]
print(f'AQ vars discretised: {available_climate}')

for col in available_climate:
    print(f'\n{col} distribution:')
    print(discr[col].value_counts().sort_index())

In [ ]:
# ── AQ bins (WHO + EEA threshold-based) — CAMS variables only ────────────────
CLIMATE_BINS = {
    'pm25': {
        'bins':   [-np.inf,  5,  10,  25,  50, np.inf],
        'labels': ['pm25_very_good', 'pm25_good', 'pm25_moderate',
                   'pm25_unhealthy', 'pm25_hazardous'],
    },
    'pm10': {
        'bins':   [-np.inf, 10,  20,  40,  80, np.inf],
        'labels': ['pm10_very_good', 'pm10_good', 'pm10_moderate',
                   'pm10_unhealthy', 'pm10_hazardous'],
    },
    'no2': {
        'bins':   [-np.inf, 10,  20,  40,  80, np.inf],
        'labels': ['no2_low', 'no2_moderate', 'no2_elevated', 'no2_high', 'no2_very_high'],
    },
    'o3': {
        'bins':   [-np.inf, 60,  100, 140, 180, np.inf],
        'labels': ['o3_low', 'o3_moderate', 'o3_elevated', 'o3_high', 'o3_very_high'],
    },
    'so2': {
        'bins':   [-np.inf, 20,  50, 100, 200, np.inf],
        'labels': ['so2_low', 'so2_moderate', 'so2_elevated', 'so2_high', 'so2_very_high'],
    },
}

discr = pd.DataFrame({'date': df['date']})
for col, spec in CLIMATE_BINS.items():
    if col in df.columns:
        discr[col] = pd.cut(df[col], bins=spec['bins'], labels=spec['labels'])

available_climate = [c for c in CLIMATE_BINS if c in df.columns]
print(f'AQ vars discretised: {available_climate}')
for col in available_climate:
    print(f'  {col}: {discr[col].value_counts().sort_index().to_dict()}')

In [5]:
# ── Health outcome bins (quantile-based per outcome) ─────────────────────────
HEALTH_CODES = [
    'code_06_breathing',
    'code_31_loss_of_consciousness',
    'code_17_falls',
    'code_09_cardiac_arrest',
    'code_10_chest_pain',
    'code_19_heart_problems',
    'code_28_stroke_tia',
    'code_12_seizure',
    'code_20_heat_cold_exposure',
    'mission_count_ems_critical',
    'mission_count_ems_critical_cpr',
]
NEGATIVE_CONTROLS = ['ctrl_27_stabbing_gunshot', 'ctrl_29_traffic_accident']

for col in HEALTH_CODES + NEGATIVE_CONTROLS:
    if col not in df.columns:
        continue
    q25, q75, q95 = df[col].quantile([0.25, 0.75, 0.95])
    short = col.replace('code_', '').replace('ctrl_', 'ctrl_').replace('mission_count_', '')
    labels = [f'{short}_low', f'{short}_normal', f'{short}_elevated', f'{short}_extreme']
    discr[col] = pd.cut(
        df[col],
        bins=[-np.inf, q25, q75, q95, np.inf],
        labels=labels
    )

available_health = [c for c in HEALTH_CODES + NEGATIVE_CONTROLS if c in discr.columns]
print(f'Health outcomes discretised: {len(available_health)}')

# Show bin distributions for a key outcome
if 'code_06_breathing' in discr.columns:
    print('\nBreathing difficulties bins:')
    print(discr['code_06_breathing'].value_counts().sort_index())

Health outcomes discretised: 13

Breathing difficulties bins:
code_06_breathing
06_breathing_low         449
06_breathing_normal      840
06_breathing_elevated    347
06_breathing_extreme      82
Name: count, dtype: int64


In [6]:
# Add season column
season_map = {12: 'winter', 1: 'winter', 2: 'winter',
               3: 'spring', 4: 'spring', 5: 'spring',
               6: 'summer', 7: 'summer', 8: 'summer',
               9: 'autumn', 10: 'autumn', 11: 'autumn'}
discr['season'] = df['date'].dt.month.map(season_map)
print('Season distribution:')
print(discr['season'].value_counts())

Season distribution:
season
autumn    455
winter    451
summer    430
spring    382
Name: count, dtype: int64


## 2. Build Transaction Dataset

Each day becomes a "transaction" — a set of items like `{hot, pm25_moderate, breathing_elevated}`.

In [7]:
def build_transactions(discr_df, climate_cols, health_cols, season=None):
    """Convert discretised dataframe into one-hot transaction matrix for FP-Growth."""
    subset = discr_df.copy()
    if season is not None:
        subset = subset[subset['season'] == season]

    all_cols = climate_cols + health_cols
    subset = subset[all_cols].dropna()

    # Convert each row to a set of item strings
    transactions = []
    for _, row in subset.iterrows():
        items = [str(v) for v in row.values if pd.notna(v)]
        transactions.append(items)

    # One-hot encode
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    ohe = pd.DataFrame(te_array, columns=te.columns_)
    return ohe, len(subset)

climate_cols  = [c for c in CLIMATE_BINS if c in discr.columns]
health_cols   = [c for c in available_health]

ohe_all, n_all = build_transactions(discr, climate_cols, health_cols)
print(f'Full dataset: {n_all} days → {ohe_all.shape[1]} unique items')

Full dataset: 1353 days → 60 unique items


## 3. Run FP-Growth + Association Rules

Parameters:
- `min_support = 0.04` (itemset must appear on ≥ 4% of days ≈ ≥50 days over 3,050)
- `min_confidence = 0.55` (if antecedent present, consequent appears 55%+ of the time)
- `min_lift = 1.3` (consequent is 30% more likely than by chance)

We filter rules where the **consequent is a health outcome** (elevated or extreme bin).

In [8]:
MIN_SUPPORT    = 0.04
MIN_CONFIDENCE = 0.55
MIN_LIFT       = 1.3

# Identify which item names correspond to elevated/extreme health outcomes
ELEVATED_ITEMS = set()
for col in health_cols:
    for level in ['elevated', 'extreme']:
        short = col.replace('code_', '').replace('ctrl_', 'ctrl_').replace('mission_count_', '')
        ELEVATED_ITEMS.add(f'{short}_{level}')

print(f'Health consequent items: {sorted(ELEVATED_ITEMS)[:8]} ...')

Health consequent items: ['06_breathing_elevated', '06_breathing_extreme', '09_cardiac_arrest_elevated', '09_cardiac_arrest_extreme', '10_chest_pain_elevated', '10_chest_pain_extreme', '12_seizure_elevated', '12_seizure_extreme'] ...


In [9]:
def run_arm(ohe, min_support=MIN_SUPPORT, min_confidence=MIN_CONFIDENCE, min_lift=MIN_LIFT):
    """Run FP-Growth and return rules with health outcome consequents."""
    freq_items = fpgrowth(ohe, min_support=min_support, use_colnames=True)
    if len(freq_items) == 0:
        return pd.DataFrame()

    rules = association_rules(freq_items, metric='confidence', min_threshold=min_confidence)
    rules = rules[rules['lift'] >= min_lift]

    # Keep only rules where consequent is a health outcome bin
    rules['consequent_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
    rules['antecedent_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    health_mask = rules['consequents'].apply(
        lambda x: any(item in ELEVATED_ITEMS for item in x)
    )
    # Also exclude rules where antecedent contains a health item
    antecedent_clean = rules['antecedents'].apply(
        lambda x: not any('_low' in i or '_normal' in i or '_elevated' in i or '_extreme' in i
                          for i in x)
    )
    return rules[health_mask & antecedent_clean].copy()

# Full dataset
rules_all = run_arm(ohe_all)
print(f'Rules (full dataset): {len(rules_all)}')
if len(rules_all) > 0:
    print(rules_all[['antecedent_str','consequent_str','support','confidence','lift']]
          .sort_values('lift', ascending=False).head(10).to_string(index=False))

Rules (full dataset): 0


## 4. Seasonal Stratification

Run ARM separately for each season to capture season-specific climate-health patterns.

In [10]:
SEASONS = ['winter', 'spring', 'summer', 'autumn']
seasonal_rules = {}

for season in SEASONS:
    ohe_s, n_s = build_transactions(discr, climate_cols, health_cols, season=season)
    # Lower min_support for seasonal subsets (fewer days)
    ms = max(0.05, 50 / n_s) if n_s > 0 else 0.05
    rules_s = run_arm(ohe_s, min_support=ms)
    seasonal_rules[season] = rules_s
    print(f'{season:8s}: {n_s:4d} days → {len(rules_s):3d} rules  (min_support={ms:.3f})')

winter  :  361 days →   0 rules  (min_support=0.139)


spring  :  290 days →   0 rules  (min_support=0.172)


summer  :  338 days →   0 rules  (min_support=0.148)


autumn  :  364 days →   0 rules  (min_support=0.137)


In [11]:
# Combine all seasonal rules with a season label
all_season_rules = []
for season, rules in seasonal_rules.items():
    if len(rules) > 0:
        r = rules.copy()
        r['season'] = season
        all_season_rules.append(r)

if all_season_rules:
    rules_combined = pd.concat([rules_all.assign(season='all')] + all_season_rules,
                                ignore_index=True)
else:
    rules_combined = rules_all.assign(season='all')

print(f'Total rules across all strata: {len(rules_combined)}')

Total rules across all strata: 0


## 5. Stability Filter — Keep Only Rules Appearing in ≥2 Strata

Rules that appear only in one stratum (e.g., only in the full dataset) may be coincidental. Stable rules appear in both the full dataset and at least one seasonal subset.

In [12]:
if len(rules_combined) > 0:
    # Count how many strata each (antecedent, consequent) pair appears in
    pair_counts = (
        rules_combined
        .groupby(['antecedent_str', 'consequent_str'])['season']
        .nunique()
        .reset_index(name='n_strata')
    )
    stable_pairs = pair_counts[pair_counts['n_strata'] >= 2]
    rules_stable = rules_combined.merge(stable_pairs[['antecedent_str','consequent_str']],
                                         on=['antecedent_str','consequent_str'])
    print(f'Rules surviving stability filter (≥2 strata): {len(rules_stable)}')
    if len(rules_stable) > 0:
        print(rules_stable[['season','antecedent_str','consequent_str',
                              'support','confidence','lift']]
              .sort_values('lift', ascending=False)
              .drop_duplicates(['antecedent_str','consequent_str'])
              .head(20).to_string(index=False))

## 6. Negative Control Validation

Rules where the consequent is a negative control (stabbing, traffic) should have lift ≈ 1. Significant lift on these would indicate a spurious climate-crime association — a red flag for confounding.

In [13]:
CONTROL_ITEMS = set()
for col in [c for c in NEGATIVE_CONTROLS if c in discr.columns]:
    short = col.replace('ctrl_', 'ctrl_')
    for level in ['elevated', 'extreme']:
        CONTROL_ITEMS.add(f'{short}_{level}')

# Mine rules with control consequents (same parameters)
def run_arm_controls(ohe):
    freq_items = fpgrowth(ohe, min_support=MIN_SUPPORT, use_colnames=True)
    if len(freq_items) == 0:
        return pd.DataFrame()
    rules = association_rules(freq_items, metric='confidence', min_threshold=0.3)
    ctrl_mask = rules['consequents'].apply(
        lambda x: any(item in CONTROL_ITEMS for item in x)
    )
    antecedent_clean = rules['antecedents'].apply(
        lambda x: not any('_low' in i or '_normal' in i or '_elevated' in i or '_extreme' in i
                          for i in x)
    )
    return rules[ctrl_mask & antecedent_clean].copy()

ctrl_rules = run_arm_controls(ohe_all)
if len(ctrl_rules) > 0:
    ctrl_rules['antecedent_str'] = ctrl_rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
    ctrl_rules['consequent_str'] = ctrl_rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
    print(f'Control rules found: {len(ctrl_rules)}')
    print(ctrl_rules[['antecedent_str','consequent_str','lift']]
          .sort_values('lift', ascending=False).head(10).to_string(index=False))
    max_ctrl_lift = ctrl_rules['lift'].max()
    if max_ctrl_lift > 1.5:
        print(f'\n⚠ WARNING: max lift on negative control = {max_ctrl_lift:.2f} (>1.5)')
        print('  This suggests climate-crime correlation — check whether seasonal control is adequate')
    else:
        print(f'\nNegative control max lift: {max_ctrl_lift:.2f} — no spurious associations detected')
else:
    print('No rules with negative control consequents at these thresholds — good.')

Control rules found: 3
        antecedent_str                             consequent_str     lift
o3_moderate, pm25_good ctrl_29_traffic_accident_elevated, no2_low 1.511823
o3_moderate, pm25_good          ctrl_29_traffic_accident_elevated 1.205259
             pm25_good          ctrl_29_traffic_accident_elevated 1.142973

⚠ WARNING: max lift on negative control = 1.51 (>1.5)
  This suggests climate-crime correlation — check whether seasonal control is adequate


## 7. Visualisation

In [14]:
# Bubble plot: support vs confidence, sized by lift, coloured by season
if 'rules_stable' in dir() and len(rules_stable) > 0:
    plot_df = (
        rules_stable
        .drop_duplicates(['antecedent_str', 'consequent_str'])
        .sort_values('lift', ascending=False)
        .head(40)
    )

    season_colors = {'all': 'gray', 'summer': 'tomato',
                     'winter': 'steelblue', 'spring': 'seagreen', 'autumn': 'darkorange'}
    colors = plot_df['season'].map(season_colors).fillna('gray')

    fig, ax = plt.subplots(figsize=(10, 6))
    sc = ax.scatter(plot_df['support'], plot_df['confidence'],
                    s=plot_df['lift'] ** 2.5 * 20,
                    c=colors, alpha=0.7, edgecolors='white', lw=0.5)

    # Annotate top rules by lift
    for _, row in plot_df.head(8).iterrows():
        label = f"{row['antecedent_str'][:30]}\n→{row['consequent_str'][:25]}"
        ax.annotate(label, (row['support'], row['confidence']),
                    fontsize=5.5, ha='left', va='bottom',
                    xytext=(4, 4), textcoords='offset points')

    legend_patches = [mpatches.Patch(color=c, label=s) for s, c in season_colors.items()]
    ax.legend(handles=legend_patches, title='Season', fontsize=8, loc='lower right')
    ax.set_xlabel('Support (fraction of days)', fontsize=10)
    ax.set_ylabel('Confidence', fontsize=10)
    ax.set_title('Association Rules: Climate → Elevated Health Demand\n(size = lift)', fontsize=11)
    ax.axhline(MIN_CONFIDENCE, color='gray', ls=':', lw=0.8)
    fig.tight_layout()
    plt.savefig(PROC_DIR / 'arm_bubble_plot.png', dpi=150, bbox_inches='tight')
    plt.show()

In [15]:
# Top rules per health outcome — bar chart of lift
if 'rules_stable' in dir() and len(rules_stable) > 0:
    unique_rules = (
        rules_stable
        .sort_values('lift', ascending=False)
        .drop_duplicates(['antecedent_str', 'consequent_str'])
    )

    # Extract outcome from consequent string
    unique_rules['outcome_short'] = unique_rules['consequent_str'].str.split('_elevated|_extreme').str[0]
    top_per_outcome = (
        unique_rules.groupby('outcome_short')
        .apply(lambda g: g.nlargest(3, 'lift'))
        .reset_index(drop=True)
    )

    if len(top_per_outcome) > 0:
        outcomes_present = top_per_outcome['outcome_short'].unique()
        n_out = len(outcomes_present)
        fig, axes = plt.subplots(1, min(n_out, 4), figsize=(5 * min(n_out, 4), 5), squeeze=False)

        for i, outcome_short in enumerate(outcomes_present[:4]):
            ax = axes[0][i]
            sub = top_per_outcome[top_per_outcome['outcome_short'] == outcome_short]
            labels = [r[:40] for r in sub['antecedent_str']]
            ax.barh(labels, sub['lift'], color='steelblue', alpha=0.8)
            ax.axvline(1, color='gray', ls='--', lw=0.8)
            ax.set_xlabel('Lift')
            ax.set_title(outcome_short.replace('_', ' '), fontsize=9)

        fig.suptitle('Top ARM Rules per Health Outcome (lift > 1 = climate increases risk)', y=1.01)
        fig.tight_layout()
        plt.savefig(PROC_DIR / 'arm_top_rules_per_outcome.png', dpi=150, bbox_inches='tight')
        plt.show()

if 'rules_combined' in dir() and len(rules_combined) > 0:
    out = PROC_DIR / 'arm_rules_cams.csv'
    export_cols = ['season', 'antecedent_str', 'consequent_str',
                   'support', 'confidence', 'lift', 'leverage', 'conviction']
    export_cols = [c for c in export_cols if c in rules_combined.columns]
    rules_combined[export_cols].sort_values('lift', ascending=False).to_csv(out, index=False)
    print(f'Saved: {out}  ({len(rules_combined)} rules)')

if 'rules_stable' in dir() and len(rules_stable) > 0:
    out_stable = PROC_DIR / 'arm_rules_stable_cams.csv'
    stable_cols = export_cols + (['n_strata'] if 'n_strata' in rules_stable.columns else [])
    rules_stable[stable_cols].sort_values('lift', ascending=False).to_csv(out_stable, index=False)
    print(f'Saved: {out_stable}  ({len(rules_stable)} stable rules)')

if 'ctrl_rules' in dir() and len(ctrl_rules) > 0:
    ctrl_rules.to_csv(PROC_DIR / 'arm_rules_cams_negative_controls.csv', index=False)
    print(f'Saved: arm_rules_cams_negative_controls.csv')

print('\nARM (CAMS-only, COVID-excluded) analysis complete.')
print('Next: notebook 06 — Bayesian Network to validate and extend these associations.')

In [16]:
if 'rules_combined' in dir() and len(rules_combined) > 0:
    out = PROC_DIR / 'arm_rules_all.csv'
    export_cols = ['season', 'antecedent_str', 'consequent_str',
                   'support', 'confidence', 'lift', 'leverage', 'conviction']
    export_cols = [c for c in export_cols if c in rules_combined.columns]
    rules_combined[export_cols].sort_values('lift', ascending=False).to_csv(out, index=False)
    print(f'Saved: {out}  ({len(rules_combined)} rules)')

if 'rules_stable' in dir() and len(rules_stable) > 0:
    out = PROC_DIR / 'arm_rules_stable.csv'
    rules_stable[export_cols + ['n_strata'] if 'n_strata' in rules_stable.columns else export_cols]\
        .sort_values('lift', ascending=False).to_csv(out, index=False)
    print(f'Saved: {out}  ({len(rules_stable)} stable rules)')

if 'ctrl_rules' in dir() and len(ctrl_rules) > 0:
    ctrl_rules.to_csv(PROC_DIR / 'arm_rules_negative_controls.csv', index=False)
    print(f'Saved: arm_rules_negative_controls.csv')

print('\nARM analysis complete.')
print('Next: notebook 06 — Bayesian Network to validate and extend these associations.')

Saved: arm_rules_negative_controls.csv

ARM analysis complete.
Next: notebook 06 — Bayesian Network to validate and extend these associations.
